# FORTRAN - pochodne typy zmiennych (wstęp do OOP)
Siódmy blok zajęć: własne struktury danych i metody powiązane z typem.

Cele zajęć:
- definiować typy złożone przez `type ... end type`
- przechowywać dane rekordowe (np. obiekty astronomiczne)
- używać tablic typów pochodnych
- rozumieć podstawę metod typu (`type-bound procedures`)

Plan spotkania:
1. Prosty typ `student`
2. Tablica rekordów
3. Typy i metody w module
4. Przykład danych astronomicznych
5. Mini-checkpoint + ćwiczenia + ściąga

Materiał pomocniczy:
- https://www.tutorialspoint.com/fortran/fortran_derived_data_types.htm


## 0) Najprostszy pochodny typ zmiennej

Typ pochodny to „własna struktura” zawierająca pola różnych typów.


In [ ]:
program student_typ
  implicit none

  type :: student
     character(len=25) :: imie
     character(len=25) :: nazwisko
     integer :: nr_albumu
     integer :: wiek
     character(len=25) :: kierunek
     logical :: aktywny
  end type student

  type(student) :: s

  s%imie = "Janina"
  s%nazwisko = "Kowalska"
  s%nr_albumu = 999999
  s%wiek = 21
  s%kierunek = "Astronomia"
  s%aktywny = .true.

  print *, trim(s%imie), trim(s%nazwisko), s%wiek
end program student_typ


## 1) Tablica rekordów

W praktyce często mamy listę obiektów tego samego typu.


In [ ]:
program tablica_studentow
  implicit none

  integer :: i

  type :: student
     character(len=25) :: imie
     character(len=25) :: kierunek
  end type student

  type(student), dimension(3) :: grupa

  grupa(1)%imie = "Janina"; grupa(1)%kierunek = "Astronomia"
  grupa(2)%imie = "Tomasz"; grupa(2)%kierunek = "Fizyka"
  grupa(3)%imie = "Marta";  grupa(3)%kierunek = "Informatyka"

  do i = 1, size(grupa)
     print *, trim(grupa(i)%imie), "->", trim(grupa(i)%kierunek)
  end do
end program tablica_studentow


## 2) Typ w module + metoda typu

`type-bound procedure` to krok w stronę programowania obiektowego:
metoda jest „przyczepiona” do typu.


In [ ]:
module shapes
  implicit none

  real(kind=8), parameter :: pi = 3.1415926535897931_8

  type :: circle
     real(kind=8) :: radius
   contains
     procedure :: area => circle_area
  end type circle

  type :: square
     real(kind=8) :: side
   contains
     procedure :: area => square_area
  end type square

contains

  function circle_area(self) result(a)
    class(circle), intent(in) :: self
    real(kind=8) :: a
    a = pi * self%radius**2
  end function circle_area

  function square_area(self) result(a)
    class(square), intent(in) :: self
    real(kind=8) :: a
    a = self%side**2
  end function square_area

end module shapes

program demo_shapes
  use shapes
  implicit none

  type(circle) :: c
  type(square) :: s

  c%radius = 0.5_8
  s%side = 0.5_8

  print *, "Pole kola:", c%area()
  print *, "Pole kwadratu:", s%area()
end program demo_shapes


## 3) Praktyczny przykład: wczytanie listy gwiazd

Poniższy przykład pokazuje, jak połączyć typ pochodny z odczytem z pliku `stars.dat`.


In [ ]:
program stars_demo
  implicit none

  type :: star
     character(len=30) :: name
     real :: ra_deg
     real :: dec_deg
     real :: mass_sun
     real :: radius_sun
     real :: teff
  end type star

  type(star), allocatable :: stars(:)
  integer :: n, i, io

  n = 0
  open(40, file='stars.dat', status='old', action='read')
  do
     read(40, *, iostat=io)
     if (io /= 0) exit
     n = n + 1
  end do
  close(40)

  allocate(stars(n))

  open(40, file='stars.dat', status='old', action='read')
  do i = 1, n
     read(40, *) stars(i)%name, stars(i)%ra_deg, stars(i)%dec_deg, &
                 stars(i)%mass_sun, stars(i)%radius_sun, stars(i)%teff
  end do
  close(40)

  print *, "Wczytano gwiazd:", n
  do i = 1, n
     write(*,'(a,1x,f7.2,1x,f7.2)') trim(stars(i)%name), stars(i)%ra_deg, stars(i)%dec_deg
  end do

  deallocate(stars)
end program stars_demo


### Mini-checkpoint

1. Czym różni się `type(student)` od zwykłej tablicy `real`?
2. Co oznacza zapis `obj%pole`?
3. Po co przenosić definicje typów do modułu?

<details>
<summary>Odpowiedzi</summary>

- `type(student)` przechowuje wiele pól różnych typów pod jedną nazwą rekordu.
- Dostęp do pola (składowej) obiektu.
- Dla reużywalności, porządku i łatwiejszego testowania.

</details>


## 4) Typowe błędy początkujących

### 1) Brak inicjalizacji pól
Część pól może mieć przypadkowe wartości.

### 2) Za krótkie `character(len=...)`
Nazwy mogą się obcinać bez ostrzeżenia.

### 3) Mieszanie logiki typu i logiki pliku
Warto rozdzielać: typy/funkcje w module, I/O w programie głównym.


## 5) Ćwiczenia

1. Dodaj pole `distance_pc` do typu `star` i wypisz gwiazdy bliższe niż 20 pc.
2. Napisz funkcję obliczającą grawitację powierzchniową z masy i promienia gwiazdy.
3. Przenieś typ `star` i funkcje pomocnicze do modułu `star_model`.


## 6) Ściąga

- Definicja typu:
```fortran
type :: nazwa
  ...
end type nazwa
```
- Deklaracja zmiennej: `type(nazwa) :: obiekt`
- Dostęp do pola: `obiekt%pole`
- Tablica rekordów: `type(nazwa), dimension(n) :: lista`
- Metoda typu: `procedure :: metoda => implementacja`
